# 08 — Predictive AHR and bounded tactical search

## Goal
Keep the native Queen/Worker/Scout network and exact Go rules. Add an action-conditioned, slowly moving target representation objective and a wider bounded adversarial search. The control and predictive arms have identical architecture and initialization; only the predictive objective is switched. Both arms evaluate the same auxiliary graph when trajectories are supplied, with coefficient zero in the control; this preserves forward random-number consumption and exercises the same backward graph.

**Inputs:** original tagged definitions from notebooks 00–06 and 08–11, and the frozen selected Go checkpoint. **Outputs:** exported model, loss, prediction diagnostics and search APIs; `reports/ahr_jepa_v3/model_validation.json`. Default Run All runs bounded CPU checks (normally under three minutes), never full training. Training checkpoints are owned by notebook 09, under `checkpoints/<run_id>/`; this notebook owns no training checkpoint. **Resume:** restart kernel and Run All; diagnostics are deterministic and overwrite only their own report.

Representations preserve the exact state’s `to_play`, history and legal mask. Every predictor step represents one actual alternating-player move. All 82 actions, including pass at index 81, are represented. Latent predictions never change rules, legal moves, scoring or proof status. A bounded search value remains UNKNOWN as a full-game claim.

## Setup

In [ ]:
from pathlib import Path
import json, nbformat
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "notebooks/00_instance_setup_and_recovery.ipynb").exists())
shared = PROJECT / "experiments/ahr_jepa_v3/00_shared.ipynb"
for cell in nbformat.read(shared, 4).cells:
    if cell.cell_type == "code" and "ahr-export" in cell.metadata.get("tags", []):
        exec(compile(cell.source, str(shared), "exec"), globals())
v3_load_upstream(PROJECT, globals())
torch.set_num_threads(1)


## Model and target representations
The projector reads all three native AHR role states. A recurrent predictor applies actual move embeddings in chronological order, including opposing replies. The target model is a separate identical network updated only by EMA. Target gradients and dropout are disabled. A per-feature variance penalty discourages collapse; diagnostics explicitly report collapse rather than inferring playing strength from prediction loss.

The prediction objective is squared distance between unit-normalized latents, summed over features (equivalently `2*(1-cosine)`), then averaged across sequences. This keeps the configured weight independent of latent dimension. Diagnostic MSE is reported per feature for comparison with persistence.

In [ ]:
from dataclasses import dataclass, asdict
from collections import Counter
import copy, hashlib, json, math, time
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

V3_MODEL_SCHEMA = "ahr_go_predictive_v3"

class V3AHR(nn.Module):
    def __init__(self, base_config=None, latent_dim=64, action_dim=32):
        super().__init__()
        if isinstance(base_config, dict):
            base_config = GoAHRConfig(**base_config)
        self.config = base_config or GoAHRConfig(queen_cycles=2)
        if self.config.queen_cycles != 2:
            raise ValueError("V3 train and deployment configuration require two Queen cycles")
        if any(type(v) is not int or v < 2 for v in (latent_dim, action_dim)):
            raise ValueError("Latent/action dimensions must be positive integers >=2")
        self.latent_dim, self.action_dim = latent_dim, action_dim
        self.v3_config = {"base_config": asdict(self.config), "latent_dim": latent_dim, "action_dim": action_dim}
        self.base = GoAHR(self.config)
        self.projector = nn.Sequential(nn.Linear(3*self.config.hidden_dim, 2*latent_dim), nn.SiLU(),
                                       nn.Linear(2*latent_dim, latent_dim), nn.LayerNorm(latent_dim))
        self.action_embedding = nn.Embedding(self.config.board_size**2+1, action_dim)
        self.predictor = nn.GRUCell(action_dim, latent_dim)
        self.prediction_norm = nn.LayerNorm(latent_dim)
        self.latent_value_head = nn.Sequential(nn.Linear(latent_dim, latent_dim), nn.SiLU(), nn.Linear(latent_dim, 1))

    def forward(self, batch, queen_cycles=2):
        return self.base(batch, queen_cycles=queen_cycles)

    def latent_from_output(self, output):
        roles = torch.cat([output["state"][name] for name in ("queen", "worker", "scout")], dim=-1)
        return self.projector(roles)

    def encode_latent(self, states, queen_cycles=2):
        positions = [GoState.from_dict(s) if isinstance(s, dict) else s for s in states]
        output = self(encode_go_batch(positions, next(self.parameters()).device), queen_cycles=queen_cycles)
        return self.latent_from_output(output)

    def predict_latent(self, z, actions):
        if z.ndim != 2 or z.shape[1] != self.latent_dim or actions.ndim != 2 or actions.shape[0] != z.shape[0]:
            raise ValueError("Prediction requires z[B,D] and actions[B,K]")
        if actions.dtype != torch.long or actions.device != z.device or actions.shape[1] < 1:
            raise ValueError("Actions must be a nonempty long tensor on the latent device")
        if bool(((actions < 0) | (actions > self.config.board_size**2)).any()):
            raise ValueError("Move outside board/pass vocabulary")
        hidden = z
        for step in range(actions.shape[1]):
            hidden = self.prediction_norm(self.predictor(self.action_embedding(actions[:, step]), hidden))
        return hidden

    def latent_value(self, z):
        return torch.tanh(self.latent_value_head(z).squeeze(-1))


def v3_update_ema(target, model, decay=.99):
    if not isinstance(target, V3AHR) or not isinstance(model, V3AHR) or target.v3_config != model.v3_config:
        raise ValueError("EMA requires identical predictive AHR configurations")
    if not isinstance(decay, (int, float)) or isinstance(decay, bool) or not math.isfinite(decay) or not 0 <= decay <= 1:
        raise ValueError("EMA decay must be finite in [0,1]")
    with torch.no_grad():
        source = dict(model.named_parameters())
        for name, parameter in target.named_parameters():
            parameter.mul_(decay).add_(source[name].detach(), alpha=1-decay)
        source_buffers = dict(model.named_buffers())
        for name, buffer in target.named_buffers():
            buffer.copy_(source_buffers[name])
    target.requires_grad_(False).eval()
    return target


def v3_validate_sequences(sequences):
    """Reject fabricated transitions, wrong player parity, truncated history and terminal overrun."""
    grouped = {}
    for sequence in sequences:
        actions = sequence["actions"]
        if not isinstance(actions, (list, tuple)) or len(actions) not in (1, 2, 4, 8):
            raise ValueError("Sequence horizon must be one of 1,2,4,8")
        state, future = GoState.from_dict(sequence["state"]), GoState.from_dict(sequence["future_state"])
        current = state
        for action in actions:
            if type(action) is not int:
                raise ValueError("Sequence actions must be integer indices")
            current = play(current, action)
        if current.to_dict() != future.to_dict():
            raise ValueError("Sequence target differs from exact chronological Go transitions")
        grouped.setdefault(len(actions), []).append((state, future, list(actions)))
    return grouped


def v3_training_loss(model, target_model, examples, sequences, use_jepa, queen_cycles=2, jepa_weight=.1):
    if not isinstance(model, V3AHR) or queen_cycles != 2:
        raise ValueError("V3 training consistently uses two Queen cycles")
    if not math.isfinite(jepa_weight) or jepa_weight < 0:
        raise ValueError("JEPA weight must be finite and nonnegative")
    if not examples and not (use_jepa and sequences):
        raise ValueError("At least supervised examples or active JEPA sequences are required")
    if not isinstance(use_jepa, bool):
        raise TypeError("use_jepa must be Boolean")
    device = next(model.parameters()).device
    # The wrapper's forward default is two cycles; the unchanged verified loss handles masks and UNKNOWN.
    if examples:
        supervised, metrics = go_learning_loss(model, examples)
    else:
        supervised = next(model.parameters()).sum()*0
        metrics = {"policy":0., "value":0., "score":0., "ownership":0., "examples":0}
    metrics.update({"jepa_enabled":use_jepa, "jepa_loss":0., "latent_variance_penalty":0.,
                    "latent_value_distillation":0., "latent_std":None, "target_latent_std":None,
                    "prediction_cosine":None, "representation_collapsed":None,
                    "sequences":len(sequences), "horizons":{}, "queen_cycles":queen_cycles,
                    "jepa_objective_weight":jepa_weight if use_jepa else 0.,
                    "auxiliary_compute_control":"active_objective" if use_jepa else "zero_weight_matched_graph"})
    total = supervised
    if sequences:
        if target_model is model or not isinstance(target_model, V3AHR) or target_model.v3_config != model.v3_config:
            raise ValueError("JEPA requires a separate identical EMA target model")
        target_model.requires_grad_(False).eval()
        grouped = v3_validate_sequences(sequences)
        prediction_terms, online_latents, target_latents, cosine_terms, value_terms = [], [], [], [], []
        for horizon, group in sorted(grouped.items()):
            starts, futures, actions = zip(*group)
            source_output = model(encode_go_batch(list(starts), device), queen_cycles=queen_cycles)
            source_z = model.latent_from_output(source_output)
            predicted = model.predict_latent(source_z, torch.tensor(actions, device=device, dtype=torch.long))
            with torch.no_grad():
                target_output = target_model(encode_go_batch(list(futures), device), queen_cycles=queen_cycles)
                expected = target_model.latent_from_output(target_output)
            squared = (F.normalize(predicted, dim=-1)-F.normalize(expected, dim=-1)).square().sum(-1)
            prediction_terms.append(squared)
            # Match value in each exact state's own-player perspective; search negates successor value.
            value_terms.append((model.latent_value(source_z)-source_output["value"].detach()).square())
            value_terms.append((model.latent_value(predicted)-target_output["value"].detach()).square())
            online_latents.append(source_z)
            target_latents.append(expected)
            cosine_terms.append(F.cosine_similarity(predicted, expected, dim=-1))
            metrics["horizons"][str(horizon)] = len(group)
        online = torch.cat(online_latents); expected = torch.cat(target_latents)
        online_std = torch.sqrt(online.var(dim=0, unbiased=False)+1e-4)
        target_std = torch.sqrt(expected.var(dim=0, unbiased=False)+1e-4)
        variance_penalty = F.relu(.5-online_std).mean() if len(online) > 1 else online.sum()*0
        prediction_loss = torch.cat(prediction_terms).mean()
        value_distillation = torch.cat(value_terms).mean()
        jepa_loss = prediction_loss + .1*variance_penalty + .1*value_distillation
        total = total + (jepa_weight if use_jepa else 0.)*jepa_loss
        metrics.update(jepa_loss=float(jepa_loss.detach()), prediction_loss=float(prediction_loss.detach()),
                       latent_variance_penalty=float(variance_penalty.detach()),
                       latent_value_distillation=float(value_distillation.detach()),
                       latent_std=float(online_std.mean().detach()), target_latent_std=float(target_std.mean()),
                       prediction_cosine=float(torch.cat(cosine_terms).mean().detach()),
                       representation_collapsed=bool(float(target_std.mean()) < .05))
    if not bool(torch.isfinite(total)):
        raise FloatingPointError("Nonfinite predictive AHR objective")
    metrics["loss"] = float(total.detach())
    return total, metrics


def v3_prediction_diagnostics(model, sequences, target_model=None, queen_cycles=2):
    target = model if target_model is None else target_model
    grouped = v3_validate_sequences(sequences)
    if not grouped:
        raise ValueError("Diagnostics require actual future sequences")
    was_training, target_training = model.training, target.training
    model.eval(); target.eval()
    result = {"target_encoder":"online" if target is model else "EMA", "by_horizon":{}, "sequences":len(sequences)}
    latents, targets = [], []
    try:
        with torch.inference_mode():
            for horizon, group in sorted(grouped.items()):
                starts, futures, actions = zip(*group)
                z = model.encode_latent(list(starts), queen_cycles)
                expected = target.encode_latent(list(futures), queen_cycles)
                predicted = model.predict_latent(z, torch.tensor(actions, device=z.device, dtype=torch.long))
                zn, en, pn = [F.normalize(x, dim=-1) for x in (z, expected, predicted)]
                result["by_horizon"][str(horizon)] = {"count":len(group),
                    "prediction_mse":float((pn-en).square().mean()),
                    "persistence_mse":float((zn-en).square().mean()),
                    "cosine":float(F.cosine_similarity(predicted, expected).mean())}
                latents.append(z); targets.append(expected)
            result["latent_std"] = float(torch.cat(latents).std(dim=0, unbiased=False).mean())
            result["target_latent_std"] = float(torch.cat(targets).std(dim=0, unbiased=False).mean())
            result["collapsed"] = result["target_latent_std"] < .05
    finally:
        model.train(was_training)
        if target is not model:
            target.train(target_training)
    return result


## Exact bounded search
Scouts include legal immediate captures, escapes from atari, and terminating passes before policy-ranked alternatives. A pass that immediately wins the game is selected after independent transition and scoring checks, even if the learned policy ranks it last. Losing terminal passes remain explicit search alternatives and are never assumed to win. Workers apply exact state transitions; the Queen iteratively deepens adversarial replies under time and node budgets. Alpha-beta pruning uses only numeric bounded estimates, never logical proof claims. Partial deeper iterations do not replace the last complete result. Final emitted moves receive the independent verifier audit.

Latent ranking is disabled by default. Enabling it requires notebook 11’s evidence validator and an explicit receipt for this exact model digest and source identities that passed held-out prediction, collapse and latency checks. The receipt must name retained, hash-verified evaluation artifacts. Validated identity is cached for the frozen actor, and a changed model or receipt invalidates that cache. Ranking uses predicted successor value and uncertainty against the actual exact successor embedding; it never creates a synthetic Go successor.

In [ ]:
@dataclass(frozen=True)
class V3SearchConfig:
    max_depth: int = 4
    branch_width: int = 8
    max_nodes: int = 192
    queen_cycles: int = 2
    temperature: float = 1.
    use_latent_ranking: bool = False
    latent_uncertainty_weight: float = .05
    reserve_fraction: float = .05
    diagnostic_allow_cycle_ablation: bool = False

    def __post_init__(self):
        if any(type(v) is not int or v < 1 for v in (self.max_depth, self.branch_width, self.max_nodes, self.queen_cycles)):
            raise ValueError("Search budgets must be positive integers")
        if type(self.diagnostic_allow_cycle_ablation) is not bool:
            raise ValueError("Cycle ablation flag must be Boolean")
        if self.queen_cycles not in (1,2) or (self.queen_cycles != 2 and not self.diagnostic_allow_cycle_ablation) or self.max_depth > 8 or self.branch_width > 82:
            raise ValueError("V3 uses two Queen cycles (explicit diagnostic permits one), depth<=8, width<=82")
        if not math.isfinite(self.temperature) or self.temperature <= 0:
            raise ValueError("Positive finite temperature required")
        if type(self.use_latent_ranking) is not bool or not math.isfinite(self.latent_uncertainty_weight) or self.latent_uncertainty_weight < 0:
            raise ValueError("Invalid latent ranking configuration")
        if not math.isfinite(self.reserve_fraction) or not 0 <= self.reserve_fraction <= .5:
            raise ValueError("Reserve fraction must be in [0,.5]")


def v3_verified_terminal_pass(state):
    """Return exact terminal utility for the player passing, without a learned estimate."""
    if state.terminal or state.consecutive_passes != 1:
        return None
    child = play(state, state.pass_action)
    if not child.terminal or not verify_go_emission(state, state.pass_action, child):
        raise AssertionError("Independent terminal-pass transition verification failed")
    score = area_score(child)
    if score != arena_reference_score(child):
        raise AssertionError("Independent terminal-pass scoring disagreement")
    relative = state.to_play*score
    return {"state":child,"utility":float((relative>0)-(relative<0)),"black_score":float(score)}


def v3_tactical_proposals(state, legal=None):
    """Exact one-ply proposals only; no claim that an escape survives future replies."""
    legal = set(legal_actions(state) if legal is None else legal)
    groups = groups_and_liberties(state.board, state.size)
    threatened = [(stones, liberties) for stones, liberties in groups
                  if state.board[next(iter(stones))] == state.to_play and len(liberties) == 1]
    candidate = set()
    for stones, liberties in groups:
        if len(liberties) == 1:
            candidate.update(liberties & legal)
    proposals = {}
    if state.consecutive_passes == 1 and state.pass_action in legal:
        # Always search a terminating pass, including a losing one, independently of policy rank.
        proposals[state.pass_action] = {"captures":0,"atari_escapes":0,"priority":-1,"terminal_pass":True}
    opponent_before = sum(stone == -state.to_play for stone in state.board)
    for action in sorted(candidate):
        child = play(state, action)
        captures = opponent_before-sum(stone == -state.to_play for stone in child.board)
        escapes = 0
        for stones, _ in threatened:
            anchor = next(iter(stones))
            if child.board[anchor] == state.to_play:
                _, liberties = group_and_liberties(child.board, state.size, anchor)
                escapes += int(len(liberties) >= 2)
        if captures or escapes:
            proposals[action] = {"captures":captures, "atari_escapes":escapes,
                                 "priority":3*captures+2*escapes}
    return proposals


def v3_choose_move(state, model, time_budget_s=1., search_config=None, *, latent_validation_gate=None):
    config = search_config or V3SearchConfig()
    if not isinstance(state, GoState) or not isinstance(model, V3AHR) or state.size != model.config.board_size:
        raise TypeError("Expected a GoState and matching V3AHR model")
    if model.training:
        raise ValueError("Search requires eval mode")
    if isinstance(time_budget_s, bool) or not isinstance(time_budget_s, (int,float)) or not math.isfinite(time_budget_s) or time_budget_s < 0:
        raise ValueError("Move budget must be finite and nonnegative")
    start = time.monotonic()
    if config.use_latent_ranking:
        gate = latent_validation_gate or {}
        model_digest = go_model_digest(model)
        if not (gate.get("schema") == "ahr_v3_latent_inference_gate_v1" and gate.get("status") == "PASS"
                and gate.get("model_sha256") == model_digest
                and all(gate.get(k) is True for k in ("heldout_prediction_improved", "collapse_check_passed", "latency_check_passed"))):
            raise ValueError("Latent inference requires passing held-out/collapse/latency gate for this exact model")
        gate_key = (model_digest, v3_digest(gate))
        if getattr(model, "_v3_validated_latent_gate", None) != gate_key:
            validator = globals().get("v3_validate_latent_gate")
            if not callable(validator):
                raise ValueError("Load notebook 11 exports to verify retained gate evidence and frozen source identities")
            validator(PROJECT, model, gate)
            model._v3_validated_latent_gate = gate_key
    reserve = min(time_budget_s, max(.002, config.reserve_fraction*time_budget_s))
    deadline = start+max(0., time_budget_s-reserve)
    device = next(model.parameters()).device
    stats = {"nodes":0, "model_calls":0, "model_positions":0, "queen_iterations":0,
             "worker_transitions":0, "scout_candidates":0, "alpha_beta_prunes":0,
             "neural_queen_cycles":0, "latent_calls":0, "tactical_candidates":0}
    costs = {"model":0., "exact_transitions":0., "tactical_proposals":0., "latent_ranking":0., "final_verification":0.}
    cache, pheromone, uncertainties = {}, {}, []
    root_tactical, root_candidates = [], []
    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize(device)
    def check():
        if time.monotonic() >= deadline:
            raise GoAHRDeadline("time_budget")
    def key(position):
        return hashlib.sha256(json.dumps(position.to_dict(), sort_keys=True, separators=(",", ":")).encode()).hexdigest()
    def transition(position, action):
        check(); before = time.monotonic(); child = play(position, action)
        costs["exact_transitions"] += time.monotonic()-before
        stats["worker_transitions"] += 1
        return child
    def evaluate(positions):
        pending = []; seen = set()
        for position in positions:
            digest = key(position)
            if digest not in cache and digest not in seen:
                pending.append((digest, position)); seen.add(digest)
        if not pending:
            return
        check(); before = time.monotonic()
        with torch.inference_mode():
            output = model(encode_go_batch([s for _,s in pending], device), queen_cycles=config.queen_cycles)
            z = model.latent_from_output(output) if config.use_latent_ranking else None
        sync(); costs["model"] += time.monotonic()-before
        stats["model_calls"] += 1; stats["model_positions"] += len(pending)
        stats["neural_queen_cycles"] += len(pending)*config.queen_cycles
        values = output["value"].detach().cpu().tolist()
        energies = output["energies"]["soft"].detach().cpu().tolist()
        for i,(digest,position) in enumerate(pending):
            cache[digest] = {"value":values[i], "energies":energies[i], "z":None if z is None else z[i].detach()}
        check()
    def ordered(position, depth, is_root=False):
        check(); digest = key(position); legal = legal_actions(position)
        before = time.monotonic(); tactics = v3_tactical_proposals(position, legal)
        costs["tactical_proposals"] += time.monotonic()-before; check()
        stats["tactical_candidates"] += len(tactics)
        tactical = sorted(tactics, key=lambda a:(-tactics[a]["priority"], cache[digest]["energies"][a], a))
        policy = sorted(legal, key=lambda a:(cache[digest]["energies"][a]-pheromone.get((digest,a),0.), a))
        width = min(len(legal), config.branch_width+depth-1)
        chosen = tactical+[a for a in policy if a not in tactics][:max(0,width-len(tactical))]
        if config.use_latent_ranking and chosen:
            children = [transition(position,a) for a in chosen]
            evaluate([child for child in children if not child.terminal])
            before = time.monotonic(); check()
            with torch.inference_mode():
                z = cache[digest]["z"].unsqueeze(0).expand(len(chosen),-1)
                predicted = model.predict_latent(z, torch.tensor(chosen, device=device, dtype=torch.long)[:,None])
                values = (-model.latent_value(predicted)).detach().cpu().tolist()
                rank = {}
                for i,(action,child) in enumerate(zip(chosen,children)):
                    if child.terminal:
                        uncertainty = 0.
                        value = -terminal_go_targets(child,child.to_play)["outcome_target"]
                    else:
                        expected = cache[key(child)]["z"]
                        uncertainty = float(1-F.cosine_similarity(predicted[i:i+1],expected[None],dim=-1)[0])
                        value = values[i]
                    rank[action] = value+config.latent_uncertainty_weight*uncertainty
                    uncertainties.append(uncertainty)
            sync(); costs["latent_ranking"] += time.monotonic()-before
            stats["latent_calls"] += 1
            chosen.sort(key=lambda a:(-rank[a],a)); check()
        stats["scout_candidates"] += len(chosen)
        if is_root:
            root_tactical[:] = tactical; root_candidates[:] = chosen
        return chosen
    def visit(position, depth, alpha=-float("inf"), beta=float("inf")):
        check()
        if stats["nodes"] >= config.max_nodes:
            raise GoAHRDeadline("node_budget")
        stats["nodes"] += 1
        if position.terminal:
            return terminal_go_targets(position,position.to_play)["outcome_target"]
        if position.consecutive_passes == 1:
            before = time.monotonic(); terminal_pass = v3_verified_terminal_pass(position)
            costs["exact_transitions"] += time.monotonic()-before
            stats["worker_transitions"] += 1; check()
            if terminal_pass["utility"] == 1.:
                return 1.
        evaluate([position]); digest = key(position)
        if depth == 0:
            return cache[digest]["value"]
        chosen = ordered(position,depth)
        children = [(action,transition(position,action)) for action in chosen]
        if depth == 1:
            evaluate([child for _,child in children if not child.terminal])
        best = -float("inf")
        for action,child in children:
            value = -visit(child,depth-1,-beta,-alpha)
            best = max(best,value); alpha = max(alpha,value)
            pheromone[(digest,action)] = float(np.clip(.8*pheromone.get((digest,action),0.)+.1*value,-2,2))
            if alpha >= beta:
                stats["alpha_beta_prunes"] += 1; break
        return best
    if state.terminal:
        return {"action":None,"policy_target":[0.]*(state.pass_action+1),
                "root_value":terminal_go_targets(state,state.to_play)["outcome_target"],
                "status":"TERMINAL","strategic_status":"UNKNOWN","elapsed_s":time.monotonic()-start,
                "reason":"already_terminal","fallback_pass":False,"completed_depth":0,"deadline_missed":False,
                "root_tactical_candidates":[],"root_candidates":[],"cost_s":costs,**stats}
    if state.consecutive_passes == 1 and time_budget_s > 0:
        before = time.monotonic(); terminal_pass = v3_verified_terminal_pass(state)
        costs["final_verification"] += time.monotonic()-before
        stats["worker_transitions"] += 1
        if terminal_pass["utility"] == 1.:
            policy = [0.]*(state.pass_action+1); policy[state.pass_action] = 1.
            elapsed = time.monotonic()-start; missed = elapsed > time_budget_s
            return {"action":state.pass_action,"policy_target":policy,"root_value":1.,
                    "status":"UNKNOWN" if missed else "LEGAL_MOVE","strategic_status":"UNKNOWN",
                    "elapsed_s":elapsed,"reason":"verified_terminal_win","completed_depth":1,
                    "deadline_missed":missed,"fallback_pass":False,"verified_legal":True,
                    "terminal_outcome_verified":True,"terminal_black_score":terminal_pass["black_score"],
                    "pheromone_entries":0,"root_tactical_candidates":[state.pass_action],"root_candidates":[state.pass_action],
                    "latent_uncertainty_mean":None,"latent_ranking_enabled":config.use_latent_ranking,
                    "cost_s":costs,"policy_target_kind":"independently_verified_terminal_win",**stats}
    selected_action = state.pass_action; root_values = None; reason = "zero_budget"; depth_done = 0; fallback = True
    if time_budget_s > 0:
        try:
            evaluate([state]); digest = key(state)
            initial_order = ordered(state,1,True)
            selected_action = initial_order[0]; fallback = False; reason = "policy_only"
            for depth in range(1,config.max_depth+1):
                check(); stats["queen_iterations"] += 1
                chosen = ordered(state,depth,True)
                children = [(a,transition(state,a)) for a in chosen]
                if depth == 1:
                    evaluate([child for _,child in children if not child.terminal])
                iteration = {}
                for action,child in children:
                    # Independent full root windows keep every policy target value comparable.
                    iteration[action] = -visit(child,depth-1)
                check(); root_values = iteration; depth_done = depth
                selected_action = max(iteration,key=lambda a:(iteration[a],-a)); reason = "bounded_search"
                for action,value in iteration.items():
                    pheromone[(digest,action)] = float(np.clip(.8*pheromone.get((digest,action),0.)+.1*value,-2,2))
        except GoAHRDeadline as cutoff:
            reason = str(cutoff)
    policy = np.zeros(state.pass_action+1,dtype=float)
    if fallback:
        policy[selected_action] = 1.; root_value = None
    elif root_values:
        moves = sorted(root_values); values = np.array([root_values[a] for a in moves])/config.temperature
        probabilities = np.exp(values-values.max()); probabilities /= probabilities.sum(); policy[moves] = probabilities
        root_value = float(max(root_values.values()))
    else:
        root = cache.get(key(state)); legal = legal_actions(state)
        if root:
            logits = -np.array([root["energies"][a] for a in legal])/config.temperature
            probabilities = np.exp(logits-logits.max()); probabilities /= probabilities.sum(); policy[list(legal)] = probabilities
            root_value = float(root["value"])
        else:
            policy[selected_action] = 1.; root_value = None
    before = time.monotonic(); child = play(state,selected_action)
    if not verify_go_emission(state,selected_action,child):
        raise AssertionError("Independent final Go emission audit failed")
    sync(); costs["final_verification"] = time.monotonic()-before
    elapsed = time.monotonic()-start; missed = elapsed > time_budget_s
    return {"action":selected_action,"policy_target":policy.tolist(),"root_value":root_value,
            "status":"UNKNOWN" if reason in ("time_budget","node_budget","zero_budget") or missed else "LEGAL_MOVE",
            "strategic_status":"UNKNOWN","elapsed_s":elapsed,"reason":"final_audit_overrun" if missed and reason=="bounded_search" else reason,
            "completed_depth":depth_done,"deadline_missed":missed,"fallback_pass":fallback,"verified_legal":True,
            "pheromone_entries":len(pheromone),"root_tactical_candidates":root_tactical,"root_candidates":root_candidates,
            "latent_uncertainty_mean":None if not uncertainties else float(np.mean(uncertainties)),
            "latent_ranking_enabled":config.use_latent_ranking,"cost_s":costs,
            "policy_target_kind":"softmax_explored_backed_up_values" if root_values else "native_energy_policy" if cache else "verified_fallback",
            **stats}


## Checks
These checks use a frozen real trained base model and exact generated Go transitions. Disposable optimizer updates test gradients; they are not a training result. Tests cover base-weight preservation, target stop gradients and EMA arithmetic, masking, UNKNOWN label rejection, pass/action order, 1/2/4/8-move horizons, legal ko history, tactical inclusion and finite move deadlines. The last table records observations; it makes no claim of improved playing strength.

In [ ]:
torch.manual_seed(8023)
BASE08, BASE_META08 = load_go_ahr(PROJECT, "go09-first-selfplay-v1", device="cpu", selection="latest")
MODEL08 = V3AHR(BASE08.config)
MODEL08.base.load_state_dict(BASE08.state_dict(), strict=True)
MODEL08.eval()
assert go_model_digest(MODEL08.base) == go_model_digest(BASE08)
TARGET08 = copy.deepcopy(MODEL08).requires_grad_(False).eval()
initial08 = initial_state(size=9)
trajectory08 = [initial08]
for action08 in (0, 80, 1, 79, 2, 78, 3, 77, 4, 76):
    trajectory08.append(play(trajectory08[-1], action08))
sequences08 = [{"state":trajectory08[start].to_dict(), "future_state":trajectory08[start+h].to_dict(),
                "actions":list((0,80,1,79,2,78,3,77,4,76)[start:start+h])}
               for h in (1,2,4,8) for start in (0,1,2) if start+h < len(trajectory08)]
assert set(v3_validate_sequences(sequences08)) == {1,2,4,8}
with torch.no_grad():
    z08 = MODEL08.encode_latent([initial08, initial08])
    different08 = MODEL08.predict_latent(z08, torch.tensor([[0,80],[80,0]], dtype=torch.long))
    assert not torch.allclose(different08[0], different08[1])
    assert torch.isfinite(MODEL08.predict_latent(z08[:1],torch.tensor([[81]],dtype=torch.long))).all()
invalid08 = copy.deepcopy(sequences08[0]); invalid08["future_state"] = initial08.to_dict()
try:
    v3_validate_sequences([invalid08]); raise AssertionError("Fabricated future accepted")
except ValueError:
    pass
policy08 = [0.]*82; policy08[0] = 1.
examples08 = [{"state":initial08.to_dict(),"policy_target":policy08,"policy_weight":1.}]
MODEL08.train(); loss08, metrics08 = v3_training_loss(MODEL08,TARGET08,examples08,sequences08,True)
loss08.backward()
assert MODEL08.projector[0].weight.grad is not None and MODEL08.predictor.weight_ih.grad is not None
assert all(p.grad is None for p in TARGET08.parameters())
assert all(p.grad is None or torch.isfinite(p.grad).all() for p in MODEL08.parameters())
old_target08 = {k:v.clone() for k,v in TARGET08.named_parameters()}
optimizer08 = torch.optim.AdamW(MODEL08.parameters(),lr=1e-5)
optimizer08.step(); optimizer08.zero_grad(set_to_none=True)
v3_update_ema(TARGET08,MODEL08,.99)
for name08,p08 in TARGET08.named_parameters():
    expected08 = old_target08[name08]*.99+dict(MODEL08.named_parameters())[name08].detach()*.01
    assert torch.allclose(p08,expected08,atol=1e-7,rtol=1e-6)
assert not TARGET08.training and not any(p.requires_grad for p in TARGET08.parameters())
only_sequences08, _ = v3_training_loss(MODEL08,TARGET08,[],sequences08,True)
only_examples08, control_metrics08 = v3_training_loss(MODEL08,TARGET08,examples08,[],False)
assert torch.isfinite(only_sequences08) and torch.isfinite(only_examples08)
unknown08 = {**examples08[0],"outcome_target":-1}
try:
    v3_training_loss(MODEL08,TARGET08,[unknown08],[],False); raise AssertionError("UNKNOWN outcome mislabeled")
except ValueError:
    pass
MODEL08.eval()
DIAGNOSTICS08 = v3_prediction_diagnostics(MODEL08,sequences08,TARGET08)
# Same forward/RNG schedule for paired arms, and zero predictive gradient in control.
control08, predictive08 = copy.deepcopy(MODEL08).train(), copy.deepcopy(MODEL08).train()
torch.manual_seed(19081)
control_loss08, parity_metrics08 = v3_training_loss(control08,TARGET08,examples08,sequences08,False)
control_rng08 = torch.get_rng_state().clone(); control_loss08.backward()
torch.manual_seed(19081)
predictive_loss08, _ = v3_training_loss(predictive08,TARGET08,examples08,sequences08,True)
predictive_rng08 = torch.get_rng_state().clone(); predictive_loss08.backward()
assert torch.equal(control_rng08,predictive_rng08)
assert all(p.grad is None or torch.count_nonzero(p.grad)==0 for name,p in control08.named_parameters() if not name.startswith("base."))
supervised08 = copy.deepcopy(MODEL08).train()
torch.manual_seed(19081)
supervised_loss08, _ = v3_training_loss(supervised08,TARGET08,examples08,[],False)
supervised_loss08.backward()
for (name08,param08),(other_name08,other_param08) in zip(control08.base.named_parameters(),supervised08.base.named_parameters()):
    assert name08 == other_name08
    if param08.grad is None or other_param08.grad is None: assert param08.grad is other_param08.grad is None
    else: assert torch.equal(param08.grad,other_param08.grad)
PARITY08 = {"forward_rng_schedule":"identical", "control_auxiliary_gradients":"zero", "control_base_gradients":"equal_supervised_only",
            "graph":"same auxiliary forward/backward, coefficient zero in control", "weight_decay":"AdamW may decay zero-gradient auxiliary parameters"}
print({"base_checkpoint":BASE_META08["checkpoint_sha256"],"base_parameters":sum(p.numel() for p in MODEL08.base.parameters()),
       "total_parameters":sum(p.numel() for p in MODEL08.parameters()),"gradient_ema_checks":"PASS", "horizons":list(DIAGNOSTICS08["by_horizon"])})


In [ ]:
board08 = [0]*81
for point08 in (1,9,19): board08[point08] = BLACK
board08[10] = WHITE
capture08 = GoState(9,tuple(board08),to_play=BLACK)
escape08 = GoState(9,tuple(-x for x in board08),to_play=BLACK)
assert 11 in v3_tactical_proposals(capture08)
assert 11 in v3_tactical_proposals(escape08)
rows08 = []
for name08,state08,budget08 in (("capture",capture08,1.),("atari_escape",escape08,1.),("opening",initial08,1.),("zero_budget",initial08,0.)):
    answer08 = v3_choose_move(state08,MODEL08,budget08)
    assert answer08["verified_legal"] and answer08["strategic_status"] == "UNKNOWN"
    assert answer08["nodes"] <= 192
    assert abs(sum(answer08["policy_target"])-1) < 1e-6
    assert all(p==0 for a,p in enumerate(answer08["policy_target"]) if a not in legal_actions(state08))
    if name08 in ("capture","atari_escape"):
        assert 11 in answer08["root_tactical_candidates"] and 11 in answer08["root_candidates"]
    if budget08:
        assert answer08["elapsed_s"] < 1.15, answer08
        assert answer08["neural_queen_cycles"] == 2*answer08["model_positions"]
    rows08.append({"case":name08,**{k:answer08[k] for k in ("action","completed_depth","elapsed_s","deadline_missed","nodes","model_positions","reason")}})
ko_board08 = list(capture08.board)
for point08 in (2,12,20): ko_board08[point08] = WHITE
ko_before08 = GoState(9,tuple(ko_board08),to_play=BLACK)
ko_after08 = play(ko_before08,11)
assert 10 not in legal_actions(ko_after08)
ko_result08 = v3_choose_move(ko_after08,MODEL08,.5)
assert ko_result08["action"] != 10 and ko_result08["policy_target"][10] == 0
try:
    v3_choose_move(initial08,MODEL08,.1,V3SearchConfig(use_latent_ranking=True)); raise AssertionError("Ungated latent ranking accepted")
except ValueError:
    pass
# Exercise enabled path with an explicitly synthetic test receipt; it is never a deployment approval.
synthetic_gate08 = {"schema":"ahr_v3_latent_inference_gate_v1","status":"PASS","model_sha256":go_model_digest(MODEL08),"heldout_prediction_improved":True,
                    "collapse_check_passed":True,"latency_check_passed":True,"purpose":"synthetic code-path test only"}
# Isolate ranking arithmetic from the later evidence validator; no gate receipt is written.
real_validator08 = globals().get("v3_validate_latent_gate")
def temporary_test_validator08(project, model, gate):
    assert gate.get("purpose") == "synthetic code-path test only"
try:
    globals()["v3_validate_latent_gate"] = temporary_test_validator08
    latent_result08 = v3_choose_move(capture08,MODEL08,1.,V3SearchConfig(max_depth=1,use_latent_ranking=True),latent_validation_gate=synthetic_gate08)
    assert latent_result08["latent_calls"] >= 1 and latent_result08["verified_legal"]
finally:
    if real_validator08 is None: globals().pop("v3_validate_latent_gate",None)
    else: globals()["v3_validate_latent_gate"] = real_validator08
    MODEL08.__dict__.pop("_v3_validated_latent_gate",None)
deep_result08 = v3_choose_move(initial08,MODEL08,4.,V3SearchConfig(branch_width=1,max_nodes=512,max_depth=4))
assert deep_result08["completed_depth"] == 4
with torch.inference_mode():
    cycle1_output08 = MODEL08(encode_go_batch([initial08]),queen_cycles=1)
    cycle2_output08 = MODEL08(encode_go_batch([initial08]),queen_cycles=2)
assert cycle1_output08["value"].shape == cycle2_output08["value"].shape == (1,)
cycle1_search08 = v3_choose_move(initial08,MODEL08,.5,V3SearchConfig(queen_cycles=1,diagnostic_allow_cycle_ablation=True,max_depth=1))
assert cycle1_search08["neural_queen_cycles"] == cycle1_search08["model_positions"]
# A terminal win must not depend on the learned beam. A losing pass must remain visible
# to search without being treated as a win. Both fixtures are exact legal trajectories.
winning_pass_state08 = play(initial08,81)
winning_pass08 = v3_choose_move(winning_pass_state08,MODEL08,1.)
assert winning_pass08["action"] == 81 and winning_pass08["terminal_outcome_verified"]
assert winning_pass08["root_value"] == 1. and winning_pass08["model_calls"] == 0
assert winning_pass08["strategic_status"] == "UNKNOWN" and not winning_pass08["deadline_missed"]
losing_pass_state08 = initial08
for action08 in (81,40,0,81):losing_pass_state08 = play(losing_pass_state08,action08)
assert v3_verified_terminal_pass(losing_pass_state08)["utility"] == -1.
assert 81 in v3_tactical_proposals(losing_pass_state08)
losing_pass08 = v3_choose_move(losing_pass_state08,MODEL08,1.,V3SearchConfig(max_depth=1,branch_width=2))
assert 81 in losing_pass08["root_candidates"] and losing_pass08["action"] != 81
assert losing_pass08["reason"] != "verified_terminal_win" and losing_pass08["verified_legal"]
TERMINAL_PASS08 = {"winning_action":winning_pass08["action"],"winning_elapsed_s":winning_pass08["elapsed_s"],
                   "winning_model_calls":winning_pass08["model_calls"],"independent_transition_and_score":"PASS",
                   "losing_pass_included":81 in losing_pass08["root_candidates"],"losing_pass_not_forced":losing_pass08["action"]!=81,
                   "internal_nodes":"Mandatory terminating-pass branch; exact verified winning pass recognized before learned leaf evaluation"}
REPORT08 = {"schema":"ahr_v3_model_search_validation_v1","status":"PASS","base_checkpoint":BASE_META08,
            "terminal_pass":TERMINAL_PASS08,"queen_cycles_training_and_search":2,"search":rows08,"prediction_diagnostics":DIAGNOSTICS08,
            "depth4_narrow_diagnostic":{"completed_depth":deep_result08["completed_depth"],"elapsed_s":deep_result08["elapsed_s"]},
            "cycle1_cycle2_diagnostic":"PASS","gradient_ema_mask_history_checks":"PASS","action_order_and_pass_checks":"PASS",
            "latent_default":"disabled","latent_gate_rejection":"PASS","synthetic_enabled_path":"PASS",
            "compute_parity":PARITY08,"training_steps":1,"training_scope":"disposable optimizer diagnostic only",
            "claim":"Implementation validation only; no improved Go strength established"}
report_path08 = PROJECT/"reports/ahr_jepa_v3/model_validation.json"
report_path08.parent.mkdir(parents=True,exist_ok=True)
atomic_json(report_path08,REPORT08)
from IPython.display import display, Markdown
columns08 = ("case", "action", "completed_depth", "elapsed_s", "deadline_missed", "nodes")
table08 = "| " + " | ".join(columns08) + " |\n|" + "---|"*len(columns08) + "\n"
for row08 in rows08:
    table08 += "| " + " | ".join(f"{row08[k]:.3f}" if isinstance(row08[k],float) else str(row08[k]) for k in columns08) + " |\n"
display(Markdown(table08))


## Handoff
Use `V3AHR(**saved_v3_config)` for exact reconstruction. Load frozen predecessor weights into `model.base`; save and restore the complete wrapper and the separate target wrapper thereafter. Call `v3_update_ema(target, model)` after each successful optimizer update. Both arms use the same architecture, two Queen cycles, data and search budgets. Only `use_jepa` changes the predictive objective. Notebook 09 owns optimizer, scheduler, RNG, replay and sampler recovery; notebook 11 owns held-out strength, prediction and latency comparisons.

The optional inference path is an experimental gated capability. A synthetic validation receipt in this notebook tests rejection and execution only; it is never persisted as deployment authorization. Full-game strength, rank and optimality remain unestablished.